In [14]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

#Load and filter to modern era (last 10 seasons)
df = pd.read_csv('epl_final.csv')
df['MatchDate'] = pd.to_datetime(df['MatchDate'])
df['SeasonYear'] = df['Season'].str.slice(0,4).astype(int)

#Last 10 completed seasons only
min_season = 2015
df_modern = df[df['SeasonYear'] >= min_season].copy().reset_index(drop=True)

print(f"Modern EPL data: {len(df_modern)} matches, {df_modern['SeasonYear'].nunique()} seasons")
print(f"Teams: {df_modern['HomeTeam'].nunique()}")

Modern EPL data: 3770 matches, 10 seasons
Teams: 34


In [15]:
#Core form metrics (EWMA - recent games 3x more important)
def ewm_form(series, span=6):  #FIXED: span=6
    return series.ewm(span=span, adjust=False).mean()

#PRIMARY FORM (goals scored/conceded)
df_modern['home_gf_form'] = df_modern.groupby('HomeTeam')['FullTimeHomeGoals'].transform(ewm_form)
df_modern['home_ga_form'] = df_modern.groupby('HomeTeam')['FullTimeAwayGoals'].transform(ewm_form)
df_modern['away_gf_form'] = df_modern.groupby('AwayTeam')['FullTimeAwayGoals'].transform(ewm_form)
df_modern['away_ga_form'] = df_modern.groupby('AwayTeam')['FullTimeHomeGoals'].transform(ewm_form)

#ADVANCED xG PROXY (shots + corners)
df_modern['home_xg_proxy'] = (df_modern['HomeShotsOnTarget'] * 0.11 + df_modern['HomeCorners'] * 0.03)
df_modern['away_xg_proxy'] = (df_modern['AwayShotsOnTarget'] * 0.11 + df_modern['AwayCorners'] * 0.03)

#CONVERSION EFFICIENCY (finishing form)
df_modern['home_conv'] = df_modern['FullTimeHomeGoals'] / df_modern['HomeShotsOnTarget'].replace(0, 0.1)
df_modern['away_conv'] = df_modern['FullTimeAwayGoals'] / df_modern['AwayShotsOnTarget'].replace(0, 0.1)
df_modern['home_conv_form'] = df_modern.groupby('HomeTeam')['home_conv'].transform(ewm_form)
df_modern['away_conv_form'] = df_modern.groupby('AwayTeam')['away_conv'].transform(ewm_form)

#FATIGUE/REST (days since last match)
df_modern['home_rest'] = df_modern.groupby('HomeTeam')['MatchDate'].diff().dt.days.fillna(7)
df_modern['away_rest'] = df_modern.groupby('AwayTeam')['MatchDate'].diff().dt.days.fillna(7)
df_modern['rest_diff'] = df_modern['home_rest'] - df_modern['away_rest']

#SHOT DOMINANCE
df_modern['shots_dom'] = ((df_modern['HomeShotsOnTarget'] - df_modern['AwayShotsOnTarget']) / 10).clip(-1, 1)


#xG PROXY (better than shots)
df_modern['home_xg_proxy'] = (df_modern['HomeShotsOnTarget'] * 0.12 +
                             df_modern['HomeCorners'] * 0.025 +
                             df_modern['HomeShots'] * 0.015)
df_modern['away_xg_proxy'] = (df_modern['AwayShotsOnTarget'] * 0.12 +
                             df_modern['AwayCorners'] * 0.025 +
                             df_modern['AwayShots'] * 0.015)
df_modern['xg_diff'] = (df_modern['home_xg_proxy'] - df_modern['away_xg_proxy']).clip(-2, 2)




# ========== TRANSFER EFFECT PROXIES  ==========
#SEASON START BOOST (new signings effect)
df_modern['season_match_num'] = df_modern.groupby(['Season', 'HomeTeam'])['MatchDate'].rank(method='first')
df_modern['season_start'] = df_modern['season_match_num'] <= 5

#SQUAD STABILITY (low shot volatility = stable roster)
df_modern['home_shot_vol'] = df_modern.groupby('HomeTeam')['HomeShots'].transform(
    lambda x: x.rolling(10, min_periods=3).std() / x.rolling(10, min_periods=3).mean())
df_modern['shot_volatility'] = df_modern['home_shot_vol'].fillna(0.2)

#RED CARD PENALTY (NEW: Most predictive)
df_modern['red_card_diff'] = (df_modern['AwayRedCards'] - df_modern['HomeRedCards']).clip(-1, 1)

print("14 features ")
print(f"Columns created: {list(df_modern.columns[-11:])}")


14 features 
Columns created: ['away_conv_form', 'home_rest', 'away_rest', 'rest_diff', 'shots_dom', 'xg_diff', 'season_match_num', 'season_start', 'home_shot_vol', 'shot_volatility', 'red_card_diff']


In [16]:
#Current 20 EPL teams only(no relegated teams)
latest_season = df_modern['Season'].max()
recent_teams = set(df_modern[df_modern['Season']==latest_season][
    ['HomeTeam', 'AwayTeam']].values.ravel())
current_teams = list(recent_teams)[:20]  # Exactly 20

# Filter to ONLY current 20 teams
df_clean = df_modern[
    df_modern['HomeTeam'].isin(current_teams) &
    df_modern['AwayTeam'].isin(current_teams)
].reset_index(drop=True)

#6-year window (stable estimates)
df_6yr = df_clean[df_clean['Season'].isin(df_clean['Season'].unique()[-6:])]
n_total = len(df_6yr)

#Chronological split
n_train = int(0.8 * n_total)
n_val = int(0.9 * n_total)

# Ensure HomeTeam and AwayTeam are categorical
df_6yr['HomeTeam'] = df_6yr['HomeTeam'].astype('category')
df_6yr['AwayTeam'] = df_6yr['AwayTeam'].astype('category')

train_data = df_6yr[:n_train].reset_index(drop=True)
val_data = df_6yr[n_train:n_val].reset_index(drop=True)
test_data = df_6yr[n_val:].reset_index(drop=True)

#Consistent categorical encoding (20 teams only)
team_categories = train_data['HomeTeam'].cat.categories
for split in [train_data, val_data, test_data]:
    split['HomeTeam'] = pd.Categorical(split['HomeTeam'], categories=team_categories)
    split['AwayTeam'] = pd.Categorical(split['AwayTeam'], categories=team_categories)
    split['home_idx'] = split['HomeTeam'].cat.codes
    split['away_idx'] = split['AwayTeam'].cat.codes

print(f"20-team, 6yr splits: Train={len(train_data)} | Val={len(val_data)} | Test={len(test_data)}")
print(f"Teams: {len(team_categories)}")


20-team, 6yr splits: Train=1347 | Val=168 | Test=169
Teams: 20


In [17]:
with pm.Model() as bhm_model:
    #Priors for stability
    mu_att = pm.Normal('mu_att', 0, 0.15)
    sigma_att = pm.HalfNormal('sigma_att', 0.25)
    mu_def = pm.Normal('mu_def', 0, 0.15)
    sigma_def = pm.HalfNormal('sigma_def', 0.25)

    #Game effects
    home_adv = pm.Normal('home_adv', 0.28, 0.08)
    transfer_boost = pm.Normal('transfer_boost', 0.12, 0.08)
    stability_effect = pm.Normal('stability_effect', -0.06, 0.05)

    #Team abilities - HEAVY shrinkage + exactly 20 teams
    team_attack = pm.Normal('team_attack', mu_att, 0.12, shape=20)
    team_defense = pm.Normal('team_defense', mu_def, 0.12, shape=20)

    #Normalize form features
    home_form = (train_data['home_gf_form'] / train_data['home_gf_form'].mean() - 1)
    home_conv = (train_data['home_conv_form'] / train_data['home_conv_form'].mean() - 1)
    away_form = (train_data['away_gf_form'] / train_data['away_gf_form'].mean() - 1)
    away_conv = (train_data['away_conv_form'] / train_data['away_conv_form'].mean() - 1)

    #Expected goals
    home_xg = (team_attack[train_data.home_idx] -
               team_defense[train_data.away_idx] +
               home_adv +
               0.25 * home_form + 0.15 * home_conv +
               transfer_boost * train_data['season_start'].astype(float) +
               stability_effect * train_data['shot_volatility'] +
               0.12 * (train_data['rest_diff']/7).clip(-1, 1) +
               0.15 * train_data['shots_dom'] +
               0.35 * train_data['xg_diff'] +
               0.30 * train_data['red_card_diff'])

    away_xg = (team_attack[train_data.away_idx] -
               team_defense[train_data.home_idx] -
               home_adv +
               0.25 * away_form - 0.15 * away_conv -
               transfer_boost * train_data['season_start'].astype(float) -
               stability_effect * train_data['shot_volatility'] -
               0.12 * (train_data['rest_diff']/7).clip(-1, 1) -
               0.15 * train_data['shots_dom'] -
               0.35 * train_data['xg_diff'] -
               0.30 * train_data['red_card_diff'])

    #Poisson likelihood
    pm.Poisson('home_goals', pm.math.exp(home_xg.clip(-1.5, 3.5)),
               observed=train_data['FullTimeHomeGoals'])
    pm.Poisson('away_goals', pm.math.exp(away_xg.clip(-1.5, 3.5)),
               observed=train_data['FullTimeAwayGoals'])

    trace = pm.sample(5000, tune=3000, target_accept=0.95, cores=4, random_seed=42)
    print("Model fit complete (20 teams, 14 features)")


Output()

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Model fit complete (20 teams, 14 features)


In [18]:
def predict_matchoutcomes(data, trace):
    preds = []

    #League averages
    lg_home_form = train_data['home_gf_form'].mean()
    lg_home_conv = train_data['home_conv_form'].mean()
    lg_away_form = train_data['away_gf_form'].mean()
    lg_away_conv = train_data['away_conv_form'].mean()

    for _, row in data.iterrows():
        #Normalize form features
        home_form_norm = (row['home_gf_form'] / lg_home_form - 1) if lg_home_form > 0 else 0
        home_conv_norm = (row['home_conv_form'] / lg_home_conv - 1) if lg_home_conv > 0 else 0
        away_form_norm = (row['away_gf_form'] / lg_away_form - 1) if lg_away_form > 0 else 0
        away_conv_norm = (row['away_conv_form'] / lg_away_conv - 1) if lg_away_conv > 0 else 0

        #home_xg_post with xG proxy
        home_xg_post = (trace.posterior['team_attack'][:,0,row.home_idx] -
                       trace.posterior['team_defense'][:,0,row.away_idx] +
                       trace.posterior['home_adv'].mean() +
                       0.25 * home_form_norm + 0.15 * home_conv_norm +
                       trace.posterior['transfer_boost'].mean() * float(row['season_start']) +
                       trace.posterior['stability_effect'].mean() * row['shot_volatility'] +
                       0.12 * np.clip(row['rest_diff']/7, -1, 1) +
                       0.15 * row['shots_dom'] +           # REDUCED from 0.20
                       0.35 * row['xg_diff'] +             # NEW: STRONGEST FEATURE
                       0.30 * row['red_card_diff'])

        #away_xg_post with xG proxy
        away_xg_post = (trace.posterior['team_attack'][:,0,row.away_idx] -
                       trace.posterior['team_defense'][:,0,row.home_idx] -
                       trace.posterior['home_adv'].mean() +
                       0.25 * away_form_norm - 0.15 * away_conv_norm -
                       trace.posterior['transfer_boost'].mean() * float(row['season_start']) -
                       trace.posterior['stability_effect'].mean() * row['shot_volatility'] -
                       0.12 * np.clip(row['rest_diff']/7, -1, 1) -
                       0.15 * row['shots_dom'] -
                       0.35 * row['xg_diff'] -
                       0.30 * row['red_card_diff'])

        #Simulate goals from posterior predictive
        home_goals_sim = np.random.poisson(lam=np.exp(float(home_xg_post.mean())), size=10000)
        away_goals_sim = np.random.poisson(lam=np.exp(float(away_xg_post.mean())), size=10000)

        preds.append([np.mean(home_goals_sim > away_goals_sim),
                     np.mean(home_goals_sim == away_goals_sim),
                     np.mean(home_goals_sim < away_goals_sim),
                     row['FullTimeResult']])

    return pd.DataFrame(preds, columns=['P(H)', 'P(D)', 'P(A)', 'Actual'])

#Generate predictions
val_preds = predict_matchoutcomes(val_data, trace)
test_preds = predict_matchoutcomes(test_data, trace)

print("Predictions complete (14 features + xG proxy)")
print(f"Val shape: {val_preds.shape}, Test shape: {test_preds.shape}")


Predictions complete (14 features + xG proxy)
Val shape: (168, 4), Test shape: (169, 4)


In [19]:

def calibrate_predictions(preds_df, val_preds):
    """Calibrate probabilities using isotonic regression on validation set"""
    #Binary calibration for each outcome separately
    calibs = {}

    for outcome, code in [('H', 0), ('D', 1), ('A', 2)]:
        y_true = (preds_df['Actual'].map({'H':0, 'D':1, 'A':2}) == code).astype(int)
        y_pred = preds_df[f'P({outcome})']

        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(val_preds[f'P({outcome})'],
                (val_preds['Actual'].map({'H':0, 'D':1, 'A':2}) == code).astype(int))

        calibs[outcome] = iso.predict(y_pred)

    #Normalize to ensure probabilities sum to 1
    calib_probs = np.stack([calibs['H'], calibs['D'], calibs['A']], axis=1)
    calib_probs = calib_probs / calib_probs.sum(axis=1, keepdims=True)

    return pd.DataFrame(calib_probs, columns=['P(H)_calib', 'P(D)_calib', 'P(A)_calib'])

#Apply calibration
print("Calibrating predictions...")
val_preds_calib = calibrate_predictions(val_preds, val_preds)
test_preds_calib = calibrate_predictions(test_preds, val_preds)

print(" Calibration complete")


Calibrating predictions...
 Calibration complete


In [21]:
def comprehensive_evaluation(pred_df):
    """Multiclass Brier + calibration - FIXED column handling"""

    if 'Actual' not in pred_df.columns:
        print("ERROR: 'Actual' column missing from predictions")
        return None

    #Map outcomes to numeric (H=0, D=1, A=2)
    actual_num = pred_df['Actual'].map({'H':0, 'D':1, 'A':2})

    #Check for missing mappings
    if actual_num.isna().any():
        print("WARNING: Unknown actual outcomes found:", pred_df[actual_num.isna()]['Actual'].unique())
        actual_num = actual_num.fillna(0)  #Default to home win

    #Probability columns
    prob_cols = ['P(H)', 'P(D)', 'P(A)']
    if not all(col in pred_df.columns for col in prob_cols):
        print("ERROR: Missing probability columns:", prob_cols)
        return None

    probs = pred_df[prob_cols].values

    #Multiclass Brier score (mean squared error across classes)
    brier_multiclass = np.mean(np.sum((probs - np.eye(3)[actual_num])**2, axis=1))

    return {
        'BrierScore': brier_multiclass,
        'LogLoss': log_loss(actual_num, probs),
        'Accuracy': accuracy_score(actual_num, probs.argmax(axis=1)),
        'PHome': probs[:,0].mean(),
        'PAway': probs[:,2].mean(),
        'Calibration_Error': np.mean(np.abs(probs.max(axis=1) -
                                (actual_num == probs.argmax(axis=1)).astype(float))),
        'HomeWin_Accuracy': accuracy_score(actual_num[actual_num==0],
                                        probs[actual_num==0,0] > 0.45),
        'AwayWin_Accuracy': accuracy_score(actual_num[actual_num==2],
                                        probs[actual_num==2,2] > 0.45)
    }

#Run evaluation
print("Evaluating model performance...")
val_results = comprehensive_evaluation(val_preds)
test_results = comprehensive_evaluation(test_preds)

if val_results is not None:
    print("=== MODEL PERFORMANCE (20 Teams, 6yr) ===")
    print(f"VALIDATION ({len(val_preds)} matches):")
    print(f"  Brier Score:      {val_results['BrierScore']:.3f}")
    print(f"  Log Loss:         {val_results['LogLoss']:.3f}")
    print(f"  Total Accuracy:   {val_results['Accuracy']:.1%}")
    print(f"  P(Home):          {val_results['PHome']:.1%}")
    print(f"  P(Away):          {val_results['PAway']:.1%}")
    print(f"  Calibration Err:  {val_results['Calibration_Error']:.3f}")

    print(f"\nTEST SET ({len(test_preds)} matches):")
    print(f"  Brier Score:      {test_results['BrierScore']:.3f}")
    print(f"  Log Loss:         {test_results['LogLoss']:.3f}")
    print(f"  Total Accuracy:   {test_results['Accuracy']:.1%}")
else:
    print("Evaluation failed - check predictions DataFrame structure")
    print("val_preds columns:", val_preds.columns.tolist())
    print("val_preds head:\n", val_preds.head())


Evaluating model performance...
=== MODEL PERFORMANCE (20 Teams, 6yr) ===
VALIDATION (168 matches):
  Brier Score:      0.535
  Log Loss:         0.902
  Total Accuracy:   55.4%
  P(Home):          46.5%
  P(Away):          31.5%
  Calibration Err:  0.429

TEST SET (169 matches):
  Brier Score:      0.534
  Log Loss:         0.904
  Total Accuracy:   58.6%


In [22]:
#Model vs Actual

def get_actual_standings(df_clean):
    """Final league table from most recent complete season (20-team only)."""
    latest_season = df_clean['Season'].max()
    season_data = df_clean[df_clean['Season'] == latest_season].copy()

    season_data['home_pts'] = season_data['FullTimeResult'].map({'H':3, 'D':1, 'A':0})
    season_data['away_pts'] = season_data['FullTimeResult'].map({'H':0, 'D':1, 'A':3})

    home = season_data.groupby('HomeTeam').agg(
        HomePoints=('home_pts', 'sum'),
        HomeGF=('FullTimeHomeGoals', 'sum'),
        HomeGA=('FullTimeAwayGoals', 'sum')
    )
    away = season_data.groupby('AwayTeam').agg(
        AwayPoints=('away_pts', 'sum'),
        AwayGF=('FullTimeAwayGoals', 'sum'),
        AwayGA=('FullTimeHomeGoals', 'sum')
    )

    tbl = home.join(away, how='outer').fillna(0)
    tbl['TotalPoints'] = tbl['HomePoints'] + tbl['AwayPoints']
    tbl['TotalGF'] = tbl['HomeGF'] + tbl['AwayGF']
    tbl['TotalGA'] = tbl['HomeGA'] + tbl['AwayGA']
    tbl['GoalDiff'] = tbl['TotalGF'] - tbl['TotalGA']  # FIXED

    tbl = tbl.sort_values(['TotalPoints', 'GoalDiff'], ascending=[False, False])
    tbl = tbl.reset_index().rename(columns={'HomeTeam': 'Team'}) # Renamed 'index' to 'HomeTeam' as per actual_tbl state
    tbl['Actual_Rank'] = np.arange(1, len(tbl) + 1)
    return tbl

#Run comparison
actual_tbl = get_actual_standings(df_clean)
model_tbl = team_stats.reset_index(drop=True).copy()
model_tbl['Model_Rank'] = np.arange(1, len(model_tbl) + 1)

comparison = (
    model_tbl[['Team', 'Overall', 'Model_Rank']]
    .merge(actual_tbl[['Team', 'TotalPoints', 'Actual_Rank']], on='Team', how='inner')
)
comparison['Rank_Diff'] = abs(comparison['Model_Rank'] - comparison['Actual_Rank'])

spearman_corr = comparison['Model_Rank'].corr(comparison['Actual_Rank'], method='spearman')

print("\n ACTUAL FINAL STANDINGS (Top 10)")
print(actual_tbl[['Team', 'TotalPoints', 'Actual_Rank']].head(10).round(0).to_string(index=False))

print(f"\n=== MODEL vs ACTUAL RANK CORRELATION ===")
print(f"Spearman: {spearman_corr:.3f}")

print("\nFULL RANK COMPARISON:")
print(comparison[['Team', 'Model_Rank', 'Actual_Rank', 'Rank_Diff', 'Overall', 'TotalPoints']].round(2))

true_top4 = actual_tbl.head(4)['Team'].tolist()
true_top6 = actual_tbl.head(6)['Team'].tolist()
true_releg = actual_tbl.tail(3)['Team'].tolist()

top4_acc = np.mean(comparison[comparison['Team'].isin(true_top4)]['Model_Rank'] <= 4)
top6_acc = np.mean(comparison[comparison['Team'].isin(true_top6)]['Model_Rank'] <= 6)
rel_acc = np.mean(comparison[comparison['Team'].isin(true_releg)]['Model_Rank'] >= 18)

print(f"\n=== COMPETITION METRICS ===")
print(f"Top4 Accuracy:  {top4_acc:.0%}")
print(f"Top6 Accuracy:  {top6_acc:.0%}")
print(f"Relegation Acc: {rel_acc:.0%}")
print(f"Mean Rank Error:{comparison['Rank_Diff'].mean():.1f}")
print(f"Perfect Matches: {sum(comparison['Rank_Diff']==0)}")


 ACTUAL FINAL STANDINGS (Top 10)
         Team  TotalPoints  Actual_Rank
    Liverpool           82            1
      Arsenal           67            2
     Man City           64            3
      Chelsea           63            4
    Newcastle           63            5
Nott'm Forest           61            6
  Aston Villa           60            7
  Bournemouth           53            8
    Brentford           52            9
     Brighton           52           10

=== MODEL vs ACTUAL RANK CORRELATION ===
Spearman: 0.195

FULL RANK COMPARISON:
              Team  Model_Rank  Actual_Rank  Rank_Diff  Overall  TotalPoints
0        Tottenham           1           16         15     0.38           38
1        Liverpool           2            1          1     0.34           82
2         Man City           3            3          0     0.33           64
3        Leicester           4           19         15     0.33           21
4        Newcastle           5            5          0     0